**`validate_occupancy_against_permits`**

Validates curated occupancy, year built, and area against Shovels building-permit evidence.

Permits are the second out-of-band evidence source, after the CHEER hand labels.

- They are not NSI-derived, not parcel-derived, and not baked into the curated output.
- Coverage varies enormously by county, so every rate is reported per county.
- Permit evidence is parcel-level, smeared onto every footprint of the parcel.
- Rows without permits mean *no evidence*, never a negative.

All permit- and survey-derived outputs go to the cache tree, never the repository.

# Configure

In [ ]:
import argparse
import re
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd

import openplaces as op

# CHEER-specific configuration lives in scripts/cheer_linkage.py. Resolve
# the repository root from any working directory (Jupyter runs notebooks
# from their own folder; scripts run from the repository root), so these
# two imports must follow the path setup.
root = Path.cwd()
while not (root / 'src' / 'openplaces').exists() and root.parent != root:
    root = root.parent
sys.path.insert(0, str(root / 'scripts'))
from cheer_linkage import (  # noqa: E402
    COLLAPSE,
    COUNTIES,
    MAX_DIST_M,
    RECIPE_ID,
    STREET_THRESHOLD,
    VALIDATION_DIR,
    load_ground_truth,
)
from openplaces.io.curator.validation import link_points_to_entities  # noqa: E402

VAL_DIR = (
    op.cfg.external_dir
    / 'US'
    / 'NC'
    / '_all'
    / 'property'
    / 'shovels'
    / '2026'
    / 'validation'
)

pd.set_option('display.width', 220)
pd.set_option('display.max_rows', 250)

In [ ]:
parser = argparse.ArgumentParser(
    description='Validate curated occupancy against Shovels permit evidence.'
)
parser.add_argument('--recipe_id', default=RECIPE_ID)
# Default None: inventory the validation directory at run time. The permit
# batch is still growing, so the county list must never be hardcoded.
parser.add_argument('--counties', nargs='*', default=None)
parser.add_argument('--out_dir', default=str(VALIDATION_DIR))
parser.add_argument('--verbose', action='store_true')

# Test arguments

In [ ]:
ARGS_TEST = '--verbose '

args_list = [x for x in ARGS_TEST.split(' ') if x != '']
args = parser.parse_args(args_list)
args

# Validate occupancy against permit evidence

## Inventory

The shovels worktree writes one file pair per county.

- Counties are discovered at run time; incomplete pairs are reported, not read.
- `n_permits` NaN means no permit evidence at all; those rows are excluded everywhere.
- `matched_via` records how the *occupancy-bearing* permits matched: `parcel_id_local` beats `address`; `none` means permits exist but none carries an occupancy type.

In [ ]:
files = sorted(VAL_DIR.glob('US-NC-*_occupancy_validation.parquet'))
kinds_by_county = {}
for f in files:
    m = re.match(r'(US-NC-\w\w)_(footprint|parcel)_occupancy_validation', f.stem)
    if m:
        kinds_by_county.setdefault(m.group(1), set()).add(m.group(2))
complete = sorted(
    c for c, kinds in kinds_by_county.items() if kinds == {'footprint', 'parcel'}
)
incomplete = sorted(set(kinds_by_county) - set(complete))
if args.counties:
    complete = [c for c in complete if c in args.counties]
print(f'complete county pairs: {len(complete)}')
if incomplete:
    print(f'incomplete (skipped, likely mid-write): {incomplete}')


def load_permits(admin_id):
    path = VAL_DIR / f'{admin_id}_footprint_occupancy_validation.parquet'
    if not path.exists():
        return None
    df = pd.read_parquet(path)
    df.index.name = 'footprint_id'
    return df


def tier(df):
    """Confidence tier for permit occupancy evidence, high to low.

    matched_via parcel_id_local beats address; mode_pct 1.0 with at least
    two occupancy-bearing permits beats a single uncorroborated one.
    """
    n_occ = pd.to_numeric(df['n_permits_with_occupancy_type'], errors='coerce')
    strong = df['occupancy_type_mode_pct'].ge(0.999) & n_occ.ge(2)
    via_id = df['matched_via'].eq('parcel_id_local')
    has = df['occupancy_type_mode'].notna()
    t = pd.Series('none', index=df.index)
    t[has & ~via_id & ~strong] = '4_addr_weak'
    t[has & ~via_id & strong] = '3_addr_strong'
    t[has & via_id & ~strong] = '2_id_weak'
    t[has & via_id & strong] = '1_id_strong'
    return t


rows = []
for county in complete:
    perm = load_permits(county)
    has_ev = perm['n_permits'].notna()
    via = perm.loc[has_ev, 'matched_via'].value_counts()
    rows.append(
        {
            'county': county,
            'n_rows': len(perm),
            'n_with_permits': int(has_ev.sum()),
            'pct': round(100 * has_ev.mean(), 1),
            'via_parcel_id': int(via.get('parcel_id_local', 0)),
            'via_address': int(via.get('address', 0)),
            'n_with_occupancy': int(perm['occupancy_type_mode'].notna().sum()),
        }
    )
inventory = pd.DataFrame(rows).sort_values('pct', ascending=False)
inventory

## Vote vs permits, per county and tier

The comparison joins on `footprint_id` (stable across branches) and scores primary footprints only.

- Permit evidence is parcel-level: a shed inherits the house's class, so secondary footprints cannot be scored against it.
- Only counties with curated output on disk and a non-trivial number of occupancy-bearing permits appear.

In [ ]:
agree_rows = []
confusions = {}
county_joins = {}
for county in complete:
    perm = load_permits(county)
    if perm is None or perm['occupancy_type_mode'].notna().sum() < 200:
        continue
    fp = op.get_entities(args.recipe_id, county, missing='ignore')
    if fp is None or fp.empty:
        continue
    cols = [
        c
        for c in [
            'occupancy_type',
            'occupancy_type_source',
            'priority_on_parcel',
            'year_built',
            'area_m2',
            'use_group_combined_parcel',
        ]
        if c in fp.columns
    ]
    joined = fp[cols].join(perm, how='inner')
    joined['vote'] = joined['occupancy_type'].astype(object).replace(COLLAPSE)
    joined['tier'] = tier(joined)
    county_joins[county] = joined
    primary = joined[
        joined['priority_on_parcel'].astype(object).isin(['primary', 'unknown'])
    ]
    for t, sub in primary[primary['tier'] != 'none'].groupby('tier'):
        both = sub[sub['vote'].notna()]
        agree_rows.append(
            {
                'county': county,
                'tier': t,
                'n': len(both),
                'agree': round((both['vote'] == both['occupancy_type_mode']).mean(), 3)
                if len(both)
                else None,
            }
        )
    strong = primary[
        primary['tier'].isin(['1_id_strong', '2_id_weak', '3_addr_strong'])
    ]
    strong = strong[strong['vote'].notna()]
    if len(strong) >= 300:
        confusions[county] = pd.crosstab(strong['vote'], strong['occupancy_type_mode'])

agreement = pd.DataFrame(agree_rows).pivot_table(
    index='county', columns='tier', values=['n', 'agree'], aggfunc='first'
)
agreement

In [ ]:
for county, table in confusions.items():
    print(f'--- {county}: vote (rows) x permit mode (cols), strong tiers ---')
    print(table.to_string())
    print()

## What the confusions say

Three disagreement blocks dominate, and each teaches something different.

**1. The Single-Family residual absorbs non-residential buildings.**

- In New Hanover, thousands of vote-SF footprints sit on parcels whose permits (and assessor strings: churches, retail, warehouses) are non-residential.
- The two-question vote has no non-residential exit: once question 1 says `single`, the SF residual overwrites even a non-residential base class.
- CHEER cannot see this error mode: the survey labels only residential structures.

**2. `Multi-Section MH` was keyword-classified as Multi-Family.**

- Onslow's vote-MF-vs-permit-MH block is one assessor string, `Multi-Section MH` (a double/triple-wide manufactured home — one dwelling).
- The reviewed `MULTI` pattern in the keyword ruleset caught it before the `\bMH\b` rule could.
- Fixed in the ruleset (`MULTI[- ]?SECTION` → Manufactured Home, reviewed); the correction lands at the next curate run.

**3. Manufactured-home precision holds up.**

- Where the vote says Manufactured Home and permits carry occupancy, permit agreement is high in every county with volume.

In [ ]:
# The Onslow block, quantified: one assessor string explains it.
on = county_joins.get('US-NC-ON')
if on is not None:
    block = on[
        on['vote'].eq('Multi-Family')
        & on['occupancy_type_mode'].eq('Manufactured Home')
    ]
    print(f'ON vote=MF & permit=MH: {len(block)} footprints')
    print(
        block['use_group_combined_parcel']
        .astype(object)
        .value_counts()
        .head(5)
        .to_string()
    )
    print()
    print(
        'vote source:',
        block['occupancy_type_source'].astype(object).value_counts().head(3).to_dict(),
    )

In [ ]:
# The New Hanover non-residential block, quantified.
ne = county_joins.get('US-NC-NE')
if ne is not None:
    nonres = [
        'Commercial',
        'Industrial',
        'Institutional',
        'Office',
        'Retail',
        'Agricultural',
        'Recreation',
    ]
    block = ne[
        ne['vote'].eq('Single-Family')
        & ne['occupancy_type_mode'].isin(nonres)
        & ne['matched_via'].eq('parcel_id_local')
    ]
    print(
        f'NE vote=SF & permit=non-residential (id-matched): {len(block)} '
        f'footprints on {block["parcel_id_local"].nunique()} parcels'
    )
    print()
    print('their assessor strings (top 10):')
    print(
        block['use_group_combined_parcel']
        .astype(object)
        .value_counts()
        .head(10)
        .to_string()
    )

## Permits vs the CHEER hand labels

The overlap is small and clustered, so it bounds what permits can certify.

- CHEER points sit in counties with thin permit coverage; the overlap concentrates in New Hanover.
- Permit evidence is parcel-level: one wrongly mapped parcel poisons every footprint on it.
- The headline example: a manufactured-home community whose 125 permits are all mapped `Office` (mode_pct 1.0) — CHEER and the vote both say Manufactured Home for all 45 points on it.
- Lesson: `mode_pct` × `n_permits` is not reliability on large multi-structure parcels. Count distinct parcels before trusting any overlap rate.

In [ ]:
frames = []
for county in COUNTIES:
    subset = load_ground_truth((county,))
    if subset.empty:
        continue
    fp = op.get_entities(args.recipe_id, county, geom=True, missing='ignore')
    if fp is None or fp.empty:
        continue
    fp = fp.reset_index()  # carry footprint_id through the linkage
    linked = link_points_to_entities(
        subset,
        fp,
        max_distance_m=MAX_DIST_M,
        street_threshold=STREET_THRESHOLD,
        admin1_id='US-NC',
    )
    if linked.empty:
        continue
    perm = load_permits(county)
    if perm is not None:
        linked = linked.merge(
            perm.reset_index(),
            left_on='footprint_id_inv',
            right_on='footprint_id',
            how='left',
        )
    frames.append(linked)
cheer = pd.concat(frames, ignore_index=True)
cheer['vote'] = cheer['occupancy_type_inv'].astype(object).replace(COLLAPSE)
cheer['tier'] = tier(cheer)

overlap = cheer[cheer['occupancy_type_mode'].notna()]
print(
    f'CHEER points linked: {len(cheer)}; with permit occupancy: '
    f'{len(overlap)} on {overlap["parcel_id_local"].nunique()} '
    'distinct parcels'
)
print()
print('permit mode vs CHEER label (all tiers):')
print(
    pd.crosstab(
        overlap['occupancy_type_canonical'], overlap['occupancy_type_mode']
    ).to_string()
)
print()
office = overlap[overlap['occupancy_type_mode'].eq('Office')]
if len(office):
    print(
        f'the Office block: {len(office)} points, '
        f'{office["parcel_id_local"].nunique()} parcel(s), '
        f'addresses {office["address"].dropna().unique()[:2].tolist()}'
    )

## The Single-Family ceiling (handoff question 3)

Is Single-Family's F1 ceiling an evidence limit or a weighting limit?

- At the CHEER points the permit overlap is too thin to arbitrate directly.
- At inventory scale, the New Hanover confusion carries the answer: large vote-MF-and-permit-SF and vote-SF-and-permit-MF blocks exist side by side, plus the non-residential leak above.
- Reading: partly evidence-limited (as the handoff guessed), but the non-residential leak is a *structural* gap no reweighting fixes — and it deflates SF precision invisibly, because CHEER never labels non-residential buildings.

In [ ]:
sf = cheer[cheer['occupancy_type_canonical'].eq('Single-Family')]
sf_wrong = sf[sf['vote'].ne('Single-Family')]
print(f'true SF: {len(sf)}; vote wrong on {len(sf_wrong)}')
print('vote said:', sf_wrong['vote'].value_counts(dropna=False).to_dict())
with_perm = sf_wrong[sf_wrong['occupancy_type_mode'].notna()]
print(
    f'of the wrong ones, {len(with_perm)} carry permit occupancy '
    '(too thin to settle the question at the points themselves)'
)

## Robeson PARDESC1 (handoff question 4.1)

Permits are absent in Robeson (28 matched rows), so the codes are tested against the 232 CHEER labels, joined spatially to the county's raw parcel file.

The handoff's letter-code reading is substantially wrong:

- `D-10` → Single-Family holds (65/67, on 66 independent parcels).
- The manufactured-home code is `C-92` (49/49 CHEER-MH), not `D-71` — but those 49 points sit on only 9 parcels.
- `C-65` is Multi-Family (29/29, 7 parcels); `E-15`/`E-80` lean Multi-Family (public/exempt housing).
- `D-71` is only weakly MH: 7/10, on 10 independent parcels.
- `V-80` (a *vacant* code, 10K parcels county-wide) carries manufactured homes at 15/17 — consistent with homes on leased land whose improvements live on separate records.

Any remap built from this should weight by independent parcels, not points.

In [ ]:
raw_rb_path = (
    op.cfg.external_dir
    / 'US'
    / 'NC'
    / 'RB'
    / '_all'
    / 'parcel'
    / 'robesoncounty'
    / '2026'
    / 'robeson_parcels.geojson'
)
if raw_rb_path.exists():
    raw_rb = gpd.read_file(raw_rb_path)
    truth_rb = load_ground_truth(('US-NC-RB',))
    pts = gpd.GeoDataFrame(
        truth_rb,
        geometry=gpd.points_from_xy(truth_rb['lon'], truth_rb['lat']),
        crs='EPSG:4326',
    ).to_crs(raw_rb.crs)
    rb = gpd.sjoin(
        pts,
        raw_rb[['PIN_NUMBER', 'PARDESC1', 'geometry']],
        how='left',
        predicate='within',
    )
    rb = rb[~rb.index.duplicated(keep='first')]
    table = pd.crosstab(rb['PARDESC1'].fillna('<none>'), rb['occupancy_type_canonical'])
    table['points'] = table.sum(axis=1)
    table['parcels'] = rb.groupby('PARDESC1')['PIN_NUMBER'].nunique()
    print(table.sort_values('points', ascending=False).head(12).to_string())
    print()
    codes = raw_rb['PARDESC1'].astype(object).str.strip()
    print(
        'county-wide counts:',
        codes.value_counts()
        .reindex(['C-92', 'C-65', 'D-10', 'D-71', 'V-80', 'E-15'])
        .to_dict(),
    )
else:
    print('raw Robeson parcel file not on disk; skipping')

## Pender zoning (handoff question 4.2)

`zoning_code` does not survive to curated parcels yet, so zones come from the *ingested* Pender parcels by spatial join (98% coverage).

The verdict is empirical and mixed:

- The `MH` zone is enriched for manufactured homes (permits: 21 MH vs 10 SF) but tiny.
- The multi-family zones `RM-CD1`/`RM-CD2` contain **zero** permit-confirmed Multi-Family — 127 Single-Family homes. Zoning states what *may* be built; here it demonstrably differs from what is.
- The broad `RP`/`RA` residential zones split roughly 3:1 SF:MH — weakly informative at best.

Zoning is therefore not worth adding as a vote input; it remains useful as descriptive context.

In [ ]:
perm_pd = load_permits('US-NC-PD')
fp_pd = op.get_entities(args.recipe_id, 'US-NC-PD', geom=True, missing='ignore')
ing_pd = op.get_entities(
    'US-NC-PD_parcel-pendercounty-2026', 'US-NC-PD', geom=True, missing='ignore'
)
if fp_pd is not None and ing_pd is not None and 'zoning_code' in ing_pd:
    joined = fp_pd[['occupancy_type', 'priority_on_parcel', 'geometry']].join(
        perm_pd, how='left'
    )
    pts = joined.copy()
    pts['geometry'] = pts.geometry.representative_point()
    zoned = gpd.sjoin(
        pts, ing_pd[['zoning_code', 'geometry']], how='left', predicate='within'
    )
    zoned = zoned[~zoned.index.duplicated(keep='first')]
    zp = zoned[
        zoned['occupancy_type_mode'].notna()
        & zoned['priority_on_parcel'].astype(object).isin(['primary', 'unknown'])
    ].copy()
    zp['vote'] = zp['occupancy_type'].astype(object).replace(COLLAPSE)
    top = zp['zoning_code'].value_counts().head(12).index
    print('permit occupancy by zone (primary footprints):')
    print(
        pd.crosstab(
            zp[zp['zoning_code'].isin(top)]['zoning_code'], zp['occupancy_type_mode']
        ).to_string()
    )
else:
    print('missing inputs for the zoning analysis; skipping')

## Year built and area (handoff question 6.4)

Permit `year_built` splits the counties into two regimes:

- Onslow and Johnston: 92–97% of primary footprints within ±1 year — the curated cascade and permits describe the same event.
- New Hanover and Pender: under 9% within ±1 year, spreads of decades — the curated `year_built` there comes from a different (likely block-median) lineage and should not be treated as parcel-grade.

Footprint area vs permit square footage is stable everywhere: the median ratio sits near 1.2 in all four counties, consistent with gross footprint vs heated area. A systematic, calibratable relationship — not a defect.

In [ ]:
for county in ['US-NC-NE', 'US-NC-ON', 'US-NC-JH', 'US-NC-PD']:
    joined = county_joins.get(county)
    if joined is None:
        continue
    primary = joined[
        joined['priority_on_parcel'].astype(object).isin(['primary', 'unknown'])
    ]
    yb = primary.dropna(subset=['year_built', 'year_built_most_recent'])
    diff = pd.to_numeric(yb['year_built']) - pd.to_numeric(yb['year_built_most_recent'])
    line = (
        f'{county}: year_built pairs={len(yb):,} '
        f'|diff|<=1yr {(diff.abs() <= 1).mean():.1%} '
        f'median {diff.median():+.0f} '
        f'p10/p90 {diff.quantile(0.1):+.0f}/{diff.quantile(0.9):+.0f}'
    )
    ar = primary.dropna(subset=['area_m2', 'area_sqft_median'])
    if len(ar):
        ratio = (
            pd.to_numeric(ar['area_m2'])
            * 10.7639
            / pd.to_numeric(ar['area_sqft_median'])
        )
        line += (
            f' | area pairs={len(ar):,} ratio p25/50/75 '
            f'{ratio.quantile(0.25):.2f}/{ratio.median():.2f}/'
            f'{ratio.quantile(0.75):.2f}'
        )
    print(line)

## Save derived artifacts

Permit- and survey-derived tables are third-party-derived, so they go to the cache tree.

In [ ]:
out_dir = Path(args.out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
inventory.to_csv(out_dir / f'{args.recipe_id}_permit-coverage.csv', index=False)
agreement.to_csv(out_dir / f'{args.recipe_id}_permit-agreement.csv')
print(f'wrote coverage + agreement tables to {out_dir}')

---
# Convert to script

*The above line and heading identify the end of the script.*

*Code below this marker will not be included in the converted `.py` script.*

In [ ]:
from openplaces.flow import convert_to_script

COMMIT = True

try:
    convert_to_script(commit=COMMIT)
except Exception as error:
    # Headless execution (nbconvert) has no notebook context to resolve
    # the caller path from; run this cell interactively to commit the
    # script. Stripped from the converted script either way.
    print(f'convert_to_script skipped: {error}')

# Test script

In [ ]:
# from openplaces.flow import test_script

# test_script(*args_list, committed=COMMIT)

# Inspect results

In [ ]:
agreement